In [36]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI


In [37]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [38]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [58]:
def select_relevant_links(url):
    openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')
    llm_model = "deepseek-r1:8b" # replace with your model name
    messages = [
        {"role": "system", "content": link_system_prompt},
        {"role": "user", "content": get_links_user_prompt(url)}
    ]

    print(f"Selecting relevant links for {url} by calling {llm_model}")

    response = openai.chat.completions.create(model=llm_model, messages=messages, response_format={"type": "json_object"})  
    result = response.choices[0].message.content

    import re
    result = re.sub(r'<think>.*?</think>', '', result, flags=re.DOTALL).strip()

    links = json.loads(result)

    if 'links' not in links:
        print(f"Warning: unexpected structure: {list(links.keys())}")
        return {"links": []}

  
    print(f"Found {len(links['links'])} relevant links")
    return links
    


In [56]:
select_relevant_links("https://www.huggingface.co")

Selecting relevant links for https://www.huggingface.co by calling deepseek-r1:8b
Found 2 relevant links


{'links': [{'type': 'company page', 'url': 'https://www.huggingface.co/'},
  'type'],
 'type': 'career page',
 'url': 'https://apply.workable.com/huggingface/'}

## Start making the brochure

In [60]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        link_url = link["url"]
        # Skip relative URLs and empty strings
        if not link_url.startswith("http"):
            print(f"Skipping invalid URL: {link_url}")
            continue
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link_url)
    return result

In [42]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

In [43]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:2_000] # Truncate if more than 2,000 characters
    return user_prompt

In [44]:
def create_brochure(company_name, url):
    openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')
    llm_model = "deepseek-r1:8b" # replace with your model name
    messages = [
        {"role": "system", "content": brochure_system_prompt},
        {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
    ]

    print(f"Creating brochure for {company_name} by calling {llm_model}")

    response = openai.chat.completions.create(model=llm_model, messages=messages)
    brochure = response.choices[0].message.content
    return brochure

In [61]:
data = create_brochure("HuggingFace", "https://huggingface.co")
display(Markdown(data))

Selecting relevant links for https://huggingface.co by calling deepseek-r1:8b
Found 0 relevant links
Creating brochure for HuggingFace by calling deepseek-r1:8b


```markdown
# The AI Community Building the Future

Hugging Face is the leading platform for open-source artificial intelligence. We host and enable collaboration on over two million models and one million applications, along with more than 500,000 datasets.

## Our Impact
Researchers, developers, and businesses collaborate on Hugging Face, accelerating AI innovation. We empower AI progress with tools and models like Transformers, Spaces, and more. Companies like InnoTech AI trust Hugging Face to power their AI applications.

## Our Culture
We foster one of the world's largest AI communities in a global environment. Driven by open-source values, we're committed to democratizing AI.

## Join Us
We're seeking passionate engineers, researchers, and community builders to continue our mission of shaping the future of artificial intelligence.

```

In [62]:
def stream_brochure_creation(company_name, url):
    openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')
    llm_model = "deepseek-r1:8b" # replace with your model name
    messages = [
        {"role": "system", "content": brochure_system_prompt},
        {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
    ]

    stream = openai.chat.completions.create(model=llm_model, messages=messages, stream=True)

    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)


In [ ]:
stream_brochure_creation("HuggingFace", "https://huggingface.co")

NameError: name 'stream_brochure' is not defined